# 1、获取大模型

# 2、使用提示词模板

In [1]:
# 确保 Windows/Japanese locale 终端也能正常打印中文输出
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

#导入 dotenv 库的 load_dotenv 函数，用于加载环境变量文件（.env）中的配置
import dotenv
from langchain_openai import ChatOpenAI
import os

dotenv.load_dotenv()  #加载当前目录下的 .env 文件

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 创建大模型实例
llm = ChatOpenAI(model="qwen2.5-coder:1.5b")  # 默认使用 qwen2.5-coder:1.5b

# 直接提供问题，并调用llm
response = llm.invoke("什么是大模型？")
print(response.content)

大模型是指规模较大的深度学习模型，通常具有成千上万的参数，能够理解和生成人类语言和表达的复杂问题。大模型是当前人工智能研究领域的重要组成部分，被广泛应用于各个领域，如语音识别、自然语言处理、计算机视觉、机器人技术等。


In [2]:
from langchain_core.prompts import ChatPromptTemplate

# 需要注意的一点是，这里需要指明具体的role，在这里是system和用户
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是世界级的技术文档编写者"),
    ("user", "{input}")  # {input}为变量
])

# 我们可以把prompt和具体llm的调用和在一起。
chain = prompt | llm
message = chain.invoke({"input": "大模型中的LangChain是什么?"})
print(message.content)

# print(type(message))

LangChain 是一个用于构建强大的语言模型的框架，它通过提供 API和工具来帮助开发者更方便地构建自然语言处理应用程序。 LangChain 使用了一个简单的对象和集合（dataclasses）的概念，这个框架的设计使得代码变得更加简洁和易于维护。它提供了多项功能，包括文本处理、问答、对话系统和机器翻译等。


# 3、 使用输出解析器

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser

# 初始化模型
llm = ChatOpenAI(model="qwen2.5-coder:1.5b")

# 创建提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是世界级的技术文档编写者。"),
    ("user", "{input}")
])

# 使用输出解析器
# output_parser = StrOutputParser()
output_parser = JsonOutputParser()

# 将其添加到上一个链中
# chain = prompt | llm
chain = prompt | llm | output_parser

# 调用它并提出同样的问题。答案是一个字符串，而不是ChatMessage
# chain.invoke({"input": "LangChain是什么?"})
print(chain.invoke({"input": "LangChain是什么? 用JSON格式回复，问题用question，回答用answer"}))

{'text': 'question：LangChain是什么？\nanswer：LangChain是一个开源的机器学习和人工智能框架，旨在提供一个完整的解决方案来应用自然语言处理，如预训练模型、上下文管理、数据加载和模型评估。它可以处理各种任务，包括文本生成、分类、问答和其他自然语言问题的分析。LangChain的设计目的是简化开发和部署复杂的人工智能应用的过程。'}


# 4、使用向量存储

In [4]:
# 导入和使用 WebBaseLoader
from langchain_community.document_loaders import WebBaseLoader
import bs4

loader = WebBaseLoader(
        web_path="https://www.gov.cn/xinwen/2020-06/01/content_5516649.htm",
        bs_kwargs=dict(parse_only=bs4.SoupStrainer(id="UCAP-CONTENT"))
    )
docs = loader.load()
print(docs)

# 对于嵌入模型，这里通过 API调用
from langchain_ollama import OllamaEmbeddings
# allama-embeddings 是一个用于生成文本嵌入的类，它使用 Ollama 模型来将文本转换为向量表示。通过指定模型名称，可以选择不同的嵌入模型来生成文本的向量表示。   
embeddings = OllamaEmbeddings(model="nomic-embed-text")


from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 使用分割器分割文档
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_documents(docs)
# 打印分割后的文档数量  
print(len(documents))
# 向量存储  embeddings 会将 documents 中的每个文本片段转换为向量，并将这些向量存储在 FAISS 向量数据库中
vector = FAISS.from_documents(documents, embeddings)

[Document(page_content='Unable to fetch https://www.gov.cn/xinwen/2020-06/01/content_5516649.htm', metadata={'source': 'https://www.gov.cn/xinwen/2020-06/01/content_5516649.htm'})]
1


# 5、RAG(检索增强生成)

In [5]:
from langchain_core.prompts import PromptTemplate

retriever = vector.as_retriever()
retriever.search_kwargs = {"k": 3}
docs = retriever.invoke("建设用地使用权是什么？")

# for i,doc in enumerate(docs):
#     print(f"⭐第{i+1}条规定：")
#     print(doc)

# 6.定义提示词模版
prompt_template = """
你是一个问答机器人。
你的任务是根据下述给定的已知信息回答用户问题。
确保你的回复完全依据下述已知信息。不要编造答案。
如果下述已知信息不足以回答用户的问题，请直接回复"我无法回答您的问题"。

已知信息:
{info}

用户问：
{question}

请用中文回答用户问题。
"""
# 7.得到提示词模版对象
template = PromptTemplate.from_template(prompt_template)

# 8.得到提示词对象
prompt = template.format(info=docs, question='建设用地使用权是什么？')

## 9. 调用LLM
response = llm.invoke(prompt)
print(response.content)

建设用地使用权，是指国家对占用土地进行依法注册登记的使用权。


# 6、使用Agent

In [6]:
from langchain.tools.retriever import create_retriever_tool

# 检索器工具
retriever_tool = create_retriever_tool(
    retriever,
    "CivilCodeRetriever",
    "搜索有关中华人民共和国民法典的信息。关于中华人民共和国民法典的任何问题，您必须使用此工具!",
)

tools = [retriever_tool]

from langchain import hub
from langchain.agents import create_openai_functions_agent
from langchain.agents import AgentExecutor

# https://smith.langchain.com/hub
prompt = hub.pull("hwchase17/openai-functions-agent")

agent = create_openai_functions_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 运行代理
agent_executor.invoke({"input": "建设用地使用权是什么"})

AIMessage(content='建设用地使用权（Land Use Right）是指土地的所有者在法律法规规定的范围内，拥有其占有、使用和处分土地的权利。包括土地所有权、土地使用权等。', additional_kwargs={})